## Business Question

Which surgical procedure categories within Island Health have the highest and most persistent wait time backlogs since 2022, and how does Island Health compare to peer health authorities?

## Plan

### Understand the dataset

Variable  |Description |
-----|-----|
FISCAL_YEAR|BC fiscal year (Apr 1 – Mar 31) [0&ndash;1]|
QUARTER|Quarter within that fiscal year (Q1=Apr–Jun, Q2=Jul–Sep, Q3=Oct–Dec, Q4=Jan–Mar)|
HEALTH_AUTHORITY|Which health authority (Fraser, Island, Vancouver Coastal, Interior, Northern, Provincial, or "All")|
HOSPITAL_NAME|Specific hospital, or "All Facilities" for rolled-up rows|
PROCEDURE_GROUP|Type of surgery (e.g. Knee Replacement, Cataract, Abdominoplasty)|
WAITING|Snapshot count of patients on the waitlist at end of that quarter|
COMPLETED|Surgeries actually performed during that quarter|
PERCENTILE_COMP_50TH|Median wait time — half of completed patients waited less than this|
PERCENTILE_COMP_90TH|90th percentile wait — 90% of completed patients waited less than this|

### Import packages

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

### Load datasets

In [4]:
df0 = pd.read_excel("../data/raw/2009_2025-quarterly-surgical_wait_times-final.xlsx")
df1 = pd.read_excel("../data/raw/2025_2026-quarterly-surgical_wait_times-q4-interim.xlsx")

In [5]:
# Concatenate two datasets
df = pd.concat([df0, df1], ignore_index = True)

### CLean numeric columns

In [6]:
df.columns = df.columns.str.strip()
for col in ['WAITING', 'COMPLETED', 'PERCENTILE_COMP_50TH', 'PERCENTILE_COMP_90TH']:
    df[col] = pd.to_numeric(
        df[col].astype("str").str.replace(',', ''), errors="coerce" # changes from 1,000 to 1000 and sets any invalid input as NaN
    )

In [7]:
'''df_rollup = df[df['HOSPITAL_NAME'] == 'All Facilities'].copy()
df_hospital_summary = df[
    (df['HOSPITAL_NAME'] != 'All Facilities') &
    (df['HEALTH_AUTHORITY'] != 'All Health Authorities') &
    (df['PROCEDURE_GROUP'] == 'All Procedures')
].copy()
df_detail = df[
    (df['HOSPITAL_NAME'] != 'All Facilities') &
    (df['HEALTH_AUTHORITY'] != 'All Health Authorities') &
    (df['PROCEDURE_GROUP'] != 'All Procedures') # include this line because we don't want 'All Procedures' appears in detailed records
].copy()'''

"df_rollup = df[df['HOSPITAL_NAME'] == 'All Facilities'].copy()\ndf_hospital_summary = df[\n    (df['HOSPITAL_NAME'] != 'All Facilities') &\n    (df['HEALTH_AUTHORITY'] != 'All Health Authorities') &\n    (df['PROCEDURE_GROUP'] == 'All Procedures')\n].copy()\ndf_detail = df[\n    (df['HOSPITAL_NAME'] != 'All Facilities') &\n    (df['HEALTH_AUTHORITY'] != 'All Health Authorities') &\n    (df['PROCEDURE_GROUP'] != 'All Procedures') # include this line because we don't want 'All Procedures' appears in detailed records\n].copy()"

### Helper columns

In [8]:
covid_years    = ['2020/21', '2021/22']
recovery_years = ['2022/23', '2023/24', '2024/25', '2025/26']

def add_helper_columns(df):
    df['backlog_ratio'] = np.round(df['WAITING'] / df['COMPLETED'], 2) #  a ratio above 1.0 means the waitlist is growing
    
    df['has_suppressed'] = np.round(df['WAITING'].isna() | df['COMPLETED'].isna(), 2)
    df['period_label'] = df['FISCAL_YEAR'] + ' ' + df['QUARTER']
    def tag_period(row):
        fy = row['FISCAL_YEAR']
        if fy == '2019/20' and row['QUARTER'] == 'Q4':
            return 'COVID Start'
        elif fy in covid_years:   return 'COVID Period'
        elif fy in recovery_years: return 'Recovery'
        else:                      return 'Pre-COVID'
    df['period_flag'] = df.apply(tag_period, axis=1)
    return df

df = add_helper_columns(df)

In [9]:
print(df.dtypes)
print(f"\nNumber of rows:           {len(df)}")
print(f"\nSuppressed in WAITING:   {df['WAITING'].isna().sum()}") # <5
print(f"Suppressed in COMPLETED: {df['COMPLETED'].isna().sum()}")

FISCAL_YEAR              object
QUARTER                  object
HEALTH_AUTHORITY         object
HOSPITAL_NAME            object
PROCEDURE_GROUP          object
WAITING                 float64
COMPLETED               float64
PERCENTILE_COMP_50TH    float64
PERCENTILE_COMP_90TH    float64
backlog_ratio           float64
has_suppressed             bool
period_label             object
period_flag              object
dtype: object

Number of rows:           215582

Suppressed in WAITING:   59685
Suppressed in COMPLETED: 61375


In [10]:
df.head(100)

,FISCAL_YEAR,QUARTER,HEALTH_AUTHORITY,HOSPITAL_NAME,PROCEDURE_GROUP,WAITING,COMPLETED,PERCENTILE_COMP_50TH,PERCENTILE_COMP_90TH,backlog_ratio,has_suppressed,period_label,period_flag
0,2009/10,Q1,All Health Authorities,All Facilities,Abdominoplasty,65.0,21.0,8.1,73.4,3.10,False,2009/10 Q1,Pre-COVID
1,2009/10,Q1,All Health Authorities,All Facilities,All Other Procedures,1552.0,1860.0,4.0,15.9,0.83,False,2009/10 Q1,Pre-COVID
2,2009/10,Q1,All Health Authorities,All Facilities,All Procedures,69587.0,58928.0,5.0,24.6,1.18,False,2009/10 Q1,Pre-COVID
3,2009/10,Q1,All Health Authorities,All Facilities,Aortic Aneurysm Repair,67.0,98.0,4.3,12.4,0.68,False,2009/10 Q1,Pre-COVID
4,2009/10,Q1,All Health Authorities,All Facilities,Appendectomy,11.0,27.0,3.9,9.5,0.41,False,2009/10 Q1,Pre-COVID
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2009/10,Q1,Fraser,Abbotsford Regional Hospital and Cancer Centre,Colostomy/Ileostomy,NaN,NaN,NaN,NaN,NaN,True,2009/10 Q1,Pre-COVID
96,2009/10,Q1,Fraser,Abbotsford Regional Hospital and Cancer Centre,Cone Biopsy,NaN,NaN,NaN,NaN,NaN,True,2009/10 Q1,Pre-COVID
97,2009/10,Q1,Fraser,Abbotsford Regional Hospital and Cancer Centre,Cyst/Ganglion Removal,NaN,NaN,NaN,NaN,NaN,True,2009/10 Q1,Pre-COVID
98,2009/10,Q1,Fraser,Abbotsford Regional Hospital and Cancer Centre,D&C and Related Surgery,71.0,46.0,3.9,11.4,1.54,False,2009/10 Q1,Pre-COVID


### Save to SQLite

In [11]:
conn = sqlite3.connect('bc_surgery.db')
df.to_sql('surgery', conn, if_exists='replace', index=False)
conn.close()
print("\nSaved to bc_surgery.db")


Saved to bc_surgery.db


In [12]:
# Export excel for Tableau
df.to_excel('../data/processed/bc_surgery.xlsx', index=False)
print('Done. All files saved.')

Done. All files saved.


### EDA

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215582 entries, 0 to 215581
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   FISCAL_YEAR           215582 non-null  object 
 1   QUARTER               215582 non-null  object 
 2   HEALTH_AUTHORITY      215582 non-null  object 
 3   HOSPITAL_NAME         215582 non-null  object 
 4   PROCEDURE_GROUP       215582 non-null  object 
 5   WAITING               155897 non-null  float64
 6   COMPLETED             154207 non-null  float64
 7   PERCENTILE_COMP_50TH  137514 non-null  float64
 8   PERCENTILE_COMP_90TH  137514 non-null  float64
 9   backlog_ratio         124078 non-null  float64
 10  has_suppressed        215582 non-null  bool   
 11  period_label          215582 non-null  object 
 12  period_flag           215582 non-null  object 
dtypes: bool(1), float64(5), object(7)
memory usage: 19.9+ MB


In [14]:
df.describe()

c:\Users\yanpu\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,WAITING,COMPLETED,PERCENTILE_COMP_50TH,PERCENTILE_COMP_90TH,backlog_ratio
count,155897.00000,154207.000000,137514.000000,137514.000000,124078.00
mean,219.01932,161.332702,8.286222,23.190045,inf
std,2034.15170,1497.687542,8.289114,18.283858,NaN
min,0.00000,0.000000,0.000000,0.000000,0.00
25%,7.00000,8.000000,3.300000,10.400000,0.72
50%,21.00000,20.000000,5.700000,19.000000,1.19
75%,74.00000,56.000000,10.100000,31.100000,1.98
max,97398.00000,77056.000000,154.700000,412.300000,inf


In [15]:
df.isna().sum()

FISCAL_YEAR                 0
QUARTER                     0
HEALTH_AUTHORITY            0
HOSPITAL_NAME               0
PROCEDURE_GROUP             0
WAITING                 59685
COMPLETED               61375
PERCENTILE_COMP_50TH    78068
PERCENTILE_COMP_90TH    78068
backlog_ratio           91504
has_suppressed              0
period_label                0
period_flag                 0
dtype: int64

In [16]:
df.duplicated().sum()

np.int64(0)

#### Province-wide waitlist trend (All Procedures)

In [17]:
conn = sqlite3.connect('bc_surgery.db')
pd.set_option('display.max_rows', None)      # Uncomment if you want to see all rows

In [18]:
pd.read_sql('''
            SELECT * 
            FROM surgery
            WHERE HEALTH_AUTHORITY = 'All Health Authorities' AND
                HOSPITAL_NAME = 'All Facilities' AND
                PROCEDURE_GROUP = 'All Procedures'
            ORDER BY FISCAL_YEAR, QUARTER
            ''', conn)

,FISCAL_YEAR,QUARTER,HEALTH_AUTHORITY,HOSPITAL_NAME,PROCEDURE_GROUP,WAITING,COMPLETED,PERCENTILE_COMP_50TH,PERCENTILE_COMP_90TH,backlog_ratio,has_suppressed,period_label,period_flag
0,2009/10,Q1,All Health Authorities,All Facilities,All Procedures,69587.0,58928.0,5.0,24.6,1.18,0,2009/10 Q1,Pre-COVID
1,2009/10,Q2,All Health Authorities,All Facilities,All Procedures,71428.0,49635.0,5.6,23.6,1.44,0,2009/10 Q2,Pre-COVID
2,2009/10,Q3,All Health Authorities,All Facilities,All Procedures,71925.0,54651.0,5.3,25.0,1.32,0,2009/10 Q3,Pre-COVID
3,2009/10,Q4,All Health Authorities,All Facilities,All Procedures,72120.0,56270.0,5.6,24.0,1.28,0,2009/10 Q4,Pre-COVID
4,2010/11,Q1,All Health Authorities,All Facilities,All Procedures,73614.0,58748.0,5.7,24.4,1.25,0,2010/11 Q1,Pre-COVID
5,2010/11,Q2,All Health Authorities,All Facilities,All Procedures,77091.0,49525.0,6.0,24.9,1.56,0,2010/11 Q2,Pre-COVID
6,2010/11,Q3,All Health Authorities,All Facilities,All Procedures,78763.0,56656.0,5.6,27.0,1.39,0,2010/11 Q3,Pre-COVID
7,2010/11,Q4,All Health Authorities,All Facilities,All Procedures,75768.0,60725.0,5.9,26.7,1.25,0,2010/11 Q4,Pre-COVID
8,2011/12,Q1,All Health Authorities,All Facilities,All Procedures,76212.0,60960.0,5.7,26.1,1.25,0,2011/12 Q1,Pre-COVID
9,2011/12,Q2,All Health Authorities,All Facilities,All Procedures,78566.0,52174.0,6.0,25.6,1.51,0,2011/12 Q2,Pre-COVID


#### Health Authority comparison (latest quarter)

In [19]:
pd.read_sql('''
            SELECT 
                FISCAL_YEAR,
                QUARTER,
                HEALTH_AUTHORITY,
                WAITING,
                COMPLETED,
                PERCENTILE_COMP_50TH,
                PERCENTILE_COMP_90TH,
                backlog_ratio
            FROM surgery
            WHERE HEALTH_AUTHORITY != 'All Health Authorities' AND
                HOSPITAL_NAME = 'All Facilities' AND
                PROCEDURE_GROUP = 'All Procedures' AND
                period_label = (SELECT MAX(period_label) FROM surgery)
            ORDER BY WAITING DESC
            ''', conn)

,FISCAL_YEAR,QUARTER,HEALTH_AUTHORITY,WAITING,COMPLETED,PERCENTILE_COMP_50TH,PERCENTILE_COMP_90TH,backlog_ratio
0,2025/26,Q4,Fraser,24346.0,19452.0,5.7,28.9,1.25
1,2025/26,Q4,Vancouver Coastal,23482.0,17133.0,5.0,31.0,1.37
2,2025/26,Q4,Vancouver Island,19034.0,14642.0,7.3,31.3,1.30
3,2025/26,Q4,Interior,18555.0,13615.0,6.9,33.0,1.36
4,2025/26,Q4,Northern,5696.0,4452.0,7.1,24.5,1.28
5,2025/26,Q4,Provincial Health Services Authority,5446.0,3430.0,7.4,31.1,1.59


#### Island Health trend over time

In [20]:
pd.read_sql('''
            SELECT 
                FISCAL_YEAR,
                QUARTER,
                HEALTH_AUTHORITY,
                WAITING,
                COMPLETED,
                PERCENTILE_COMP_50TH,
                PERCENTILE_COMP_90TH,
                backlog_ratio,
                period_flag
            FROM surgery
            WHERE HEALTH_AUTHORITY = 'Vancouver Island'
                AND HOSPITAL_NAME = 'All Facilities'
                AND PROCEDURE_GROUP = 'All Procedures'
            ORDER BY FISCAL_YEAR;
            ''', conn)

,FISCAL_YEAR,QUARTER,HEALTH_AUTHORITY,WAITING,COMPLETED,PERCENTILE_COMP_50TH,PERCENTILE_COMP_90TH,backlog_ratio,period_flag
0,2009/10,Q1,Vancouver Island,12787.0,11679.0,5.0,22.0,1.09,Pre-COVID
1,2009/10,Q2,Vancouver Island,13444.0,9363.0,6.3,22.0,1.44,Pre-COVID
2,2009/10,Q3,Vancouver Island,13774.0,11157.0,6.1,23.6,1.23,Pre-COVID
3,2009/10,Q4,Vancouver Island,13821.0,11606.0,6.1,21.1,1.19,Pre-COVID
4,2010/11,Q1,Vancouver Island,14122.0,12178.0,5.9,22.0,1.16,Pre-COVID
5,2010/11,Q2,Vancouver Island,15209.0,9849.0,6.9,23.3,1.54,Pre-COVID
6,2010/11,Q3,Vancouver Island,15195.0,11651.0,6.6,27.4,1.30,Pre-COVID
7,2010/11,Q4,Vancouver Island,14478.0,12590.0,6.7,26.3,1.15,Pre-COVID
8,2011/12,Q1,Vancouver Island,14506.0,12235.0,6.9,25.0,1.19,Pre-COVID
9,2011/12,Q2,Vancouver Island,15351.0,10183.0,7.1,24.9,1.51,Pre-COVID


#### Top 15 procedures by wait time (province, current quarter)

In [21]:
pd.read_sql('''
            SELECT 
                FISCAL_YEAR,
                QUARTER,
                PROCEDURE_GROUP,
                WAITING,
                COMPLETED,
                PERCENTILE_COMP_50TH,
                PERCENTILE_COMP_90TH,
                backlog_ratio
            FROM surgery
            WHERE HEALTH_AUTHORITY = 'All Health Authorities' AND
                HOSPITAL_NAME = 'All Facilities' AND
                PROCEDURE_GROUP != 'All Procedures' AND
                period_label = (SELECT MAX(period_label) FROM surgery)
            ORDER BY WAITING DESC
            LIMIT 15;
            ''', conn)

,FISCAL_YEAR,QUARTER,PROCEDURE_GROUP,WAITING,COMPLETED,PERCENTILE_COMP_50TH,PERCENTILE_COMP_90TH,backlog_ratio
0,2025/26,Q4,Cataract Surgery,18460.0,19305.0,5.4,21.7,0.96
1,2025/26,Q4,Knee Replacement,9875.0,3395.0,19.7,55.0,2.91
2,2025/26,Q4,Hip Replacement,5145.0,2226.0,15.9,51.3,2.31
3,2025/26,Q4,Hernia Repair - Abdominal,4645.0,3728.0,7.7,30.0,1.25
4,2025/26,Q4,Uterine Surgery,4587.0,4521.0,7.7,29.4,1.01
5,2025/26,Q4,Nasal Surgery,3599.0,926.0,24.1,61.1,3.89
6,2025/26,Q4,Dental Surgery,3373.0,1963.0,9.9,34.7,1.72
7,2025/26,Q4,Other Eye Surgery,2651.0,1884.0,5.9,32.4,1.41
8,2025/26,Q4,All Other Procedures,2435.0,2352.0,4.6,20.0,1.04
9,2025/26,Q4,Other Orthopaedic Surgery,2387.0,1661.0,6.1,31.4,1.44


#### Is the gap between 50th and 90th percentile growing?

In [22]:
pd.read_sql('''
            SELECT
                FISCAL_YEAR,
                QUARTER,
                period_label,
                period_flag,
                PERCENTILE_COMP_50TH,
                PERCENTILE_COMP_90TH,
                ROUND(PERCENTILE_COMP_90TH - PERCENTILE_COMP_50TH, 1) AS percentile_gap
            FROM surgery
            WHERE HEALTH_AUTHORITY = 'All Health Authorities'
            AND HOSPITAL_NAME    = 'All Facilities'
            AND PROCEDURE_GROUP  = 'All Procedures'
            ORDER BY FISCAL_YEAR, QUARTER;
            ''', conn)

,FISCAL_YEAR,QUARTER,period_label,period_flag,PERCENTILE_COMP_50TH,PERCENTILE_COMP_90TH,percentile_gap
0,2009/10,Q1,2009/10 Q1,Pre-COVID,5.0,24.6,19.6
1,2009/10,Q2,2009/10 Q2,Pre-COVID,5.6,23.6,18.0
2,2009/10,Q3,2009/10 Q3,Pre-COVID,5.3,25.0,19.7
3,2009/10,Q4,2009/10 Q4,Pre-COVID,5.6,24.0,18.4
4,2010/11,Q1,2010/11 Q1,Pre-COVID,5.7,24.4,18.7
5,2010/11,Q2,2010/11 Q2,Pre-COVID,6.0,24.9,18.9
6,2010/11,Q3,2010/11 Q3,Pre-COVID,5.6,27.0,21.4
7,2010/11,Q4,2010/11 Q4,Pre-COVID,5.9,26.7,20.8
8,2011/12,Q1,2011/12 Q1,Pre-COVID,5.7,26.1,20.4
9,2011/12,Q2,2011/12 Q2,Pre-COVID,6.0,25.6,19.6


#### Island Health hospital comparison (latest quarter)

In [23]:
pd.read_sql('''
            SELECT
                HOSPITAL_NAME,
                WAITING,
                COMPLETED,
                PERCENTILE_COMP_50TH,
                PERCENTILE_COMP_90TH,
                backlog_ratio
            FROM surgery
            WHERE HEALTH_AUTHORITY = 'Vancouver Island'
            AND HOSPITAL_NAME    != 'All Facilities'
            AND PROCEDURE_GROUP  = 'All Procedures'
            AND period_label     = (SELECT MAX(period_label) FROM surgery)
            ORDER BY WAITING DESC;
            ''', conn)

,HOSPITAL_NAME,WAITING,COMPLETED,PERCENTILE_COMP_50TH,PERCENTILE_COMP_90TH,backlog_ratio
0,Royal Jubilee Hospital,3871.0,3560.0,5.9,26.9,1.09
1,Victoria General Hospital,3732.0,2033.0,6.9,33.5,1.84
2,Nanaimo Regional General Hospital,3052.0,1957.0,7.1,49.3,1.56
3,"North Island Hospital, Comox Valley",2232.0,1874.0,10.1,31.6,1.19
4,Centre Island Surgical Centre,1604.0,1389.0,10.1,27.7,1.15
5,Cowichan District Hospital,1431.0,1238.0,6.1,16.6,1.16
6,"North Island Hospital, Campbell River and Dist...",1149.0,684.0,7.6,34.5,1.68
7,South Island Surgical Centre,1011.0,656.0,8.9,42.4,1.54
8,Saanich Peninsula Hospital,771.0,1075.0,7.4,27.2,0.72
9,West Coast General Hospital,181.0,176.0,4.3,17.1,1.03


In [24]:
conn.close()